# FoodLens 成员 C：Colab NVIDIA GPU 集成测试

本 Notebook 调用仓库 Python 模块，不复制训练循环。默认仅使用 8 张训练图片、执行 1 步后恢复到总计 2 步；不会自动启动全量训练。所有结果须来自实际运行，当前文件不附带任何模型成绩。

先在 Colab 选择 GPU 运行时。本文件及所调用模块目前仅提交本地；**经你检查并手动推送成员 C 分支后**，GitHub 克隆才能取得新版代码。Notebook 会检查所需文件，版本过旧就停止，不会改用 main。

上传两个 ZIP 到 `MyDrive/FoodLens/dataset/annotations.zip` 与 `MyDrive/FoodLens/dataset/images.zip`。Notebook 将它们复制到本地并解压为 `/content/foodlens_data/raw/annotations/` 和 `/content/foodlens_data/images/{train,val,test}/`。不改写 Drive 原件，不在训练中使用 val/test 标签。


In [ ]:
import os, sys, json, subprocess, platform, uuid
from pathlib import Path
from datetime import datetime, timezone

if not Path("/content").is_dir() or "google.colab" not in sys.modules and not os.environ.get("COLAB_RELEASE_TAG"):
    try:
        import google.colab
    except ImportError as exc:
        raise RuntimeError("请在 Google Colab 托管 GPU 运行时运行；不支持本地训练") from exc
import torch
assert torch.cuda.is_available(), "请先选择 NVIDIA GPU 运行时；这里不会退回 CPU 训练"
print(platform.python_version(), torch.__version__, torch.cuda.get_device_name(0))


## 获取成员 C 分支代码
先在推送后用 `git rev-parse HEAD` 取得最终完整 40 位 SHA，填写下方 EXPECTED_CODE_COMMIT（不能填旧的 6a8ac50）。留空、缩写或实际分支 HEAD 不一致都会停止；不会自动 checkout 其他版本或 main。已有目录只读检查，不自动拉取、切换、覆盖或清理。若目录版本旧，请在新运行时重试，或自行处理目录。所有命令均检查退出状态。


In [ ]:
BRANCH = "feat/member-c-diet-record"
REPO_URL = "https://github.com/liuxy2006/FoodLens.git"
REPO = Path("/content/FoodLens-member-c")
EXPECTED_CODE_COMMIT = ""  # 必填：推送本次最终提交后，粘贴 git rev-parse HEAD 的完整 40 位 SHA
import re
if not re.fullmatch(r"[0-9a-fA-F]{40}", EXPECTED_CODE_COMMIT):
    raise ValueError("请填写审核并推送后的最终完整 40 位 commit SHA；不要填写旧提交或短 SHA")
EXPECTED_CODE_COMMIT = EXPECTED_CODE_COMMIT.lower()
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "branch", "--show-current"], text=True).strip() == BRANCH
assert subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip() == REPO_URL
assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "代码目录有修改，先人工检查"
for relative in ["task2_diet_record/checkpoint.py", "task2_diet_record/integration.py", "colab/member_c_train.ipynb", "task2_diet_record/prepare_colab_data.py", "task2_diet_record/colab_dependencies.py"]:
    assert (REPO / relative).is_file(), "远程分支尚无第二阶段代码；请审核后自行推送该分支，再开新运行时"
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert CODE_COMMIT == EXPECTED_CODE_COMMIT, "代码版本与预期不一致"
os.chdir(REPO)
print("branch:", BRANCH, "commit:", CODE_COMMIT)
def run_module(module, *arguments):
    subprocess.run([sys.executable, "-m", module, *map(str, arguments)], cwd=REPO, check=True)


## 安装依赖
保留 Colab 预装的 CUDA PyTorch：安装前后在新进程读取 torch.__version__、torch.version.cuda、torch.cuda.is_available()；用精确约束锁定已安装的 torch/torchvision/torchaudio/triton，并先审查 pip dry-run 清单。若计划替换 GPU 运行时、安装 Windows/XPU 依赖，或安装后 CUDA 不可用，就停止。固定现有框架依赖，记录实际环境。若安装后提示需要重启，请重启会话后从头执行；已满足的依赖不会重复安装。


In [ ]:
# 仅在 Colab 执行：安装模块将保存 before/after、约束和 pip 计划。
INSTALL_REPORT = Path("/content/foodlens_install_" + uuid.uuid4().hex[:8])
run_module("task2_diet_record.colab_dependencies", "--output", INSTALL_REPORT)


## 挂载 Drive、设置路径和记录环境
数据只需以两个 ZIP 上传到 Drive；下一格自动识别 ZIP 层级、复制并解压到 Colab 本地。已有本地完整数据校验通过后直接复用；不完整/不一致数据会明确报错，不自动覆盖或清理。输出目录使用唯一运行 ID，断线后的恢复必须指定旧断点路径，不能覆盖旧实验。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FoodLens")
DATASET_DIR = DRIVE_ROOT / "dataset"
DATA_ROOT = Path("/content/foodlens_data")
ARCHIVE_CACHE = Path("/content/foodlens_archives")
REVISION = "7e3e67edbbed1bf9888184d9df282b700a323964"
MODEL_DIR = DRIVE_ROOT / "models" / ("SmolVLM-256M-Instruct-" + REVISION[:8])
RESUME_CHECKPOINT = ""  # 跨会话恢复时填写旧的 Drive checkpoint 目录
RESUME_TOTAL_STEPS = 2
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
RUN_ROOT = DRIVE_ROOT / "outputs" / ("member_c_" + RUN_ID)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
os.environ["HF_HOME"] = str(DRIVE_ROOT / "cache" / "huggingface")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import shutil
shutil.copytree(INSTALL_REPORT, RUN_ROOT / "dependency_install")
(RUN_ROOT / "source.json").write_text(json.dumps({"branch": BRANCH, "commit": CODE_COMMIT, "model_revision": REVISION}, indent=2))
(RUN_ROOT / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
(RUN_ROOT / "nvidia_smi.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
run_module("task2_diet_record.device_probe", "--output", RUN_ROOT / "device_probe.json")
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
print("所有结果保存至:", RUN_ROOT)


In [ ]:
from task2_diet_record.prepare_colab_data import prepare_data
preparation = prepare_data(DATASET_DIR, DATA_ROOT, ARCHIVE_CACHE)
assert Path(preparation["data_root"]) == DATA_ROOT
print(json.dumps(preparation, ensure_ascii=False, indent=2))
(RUN_ROOT / "data_preparation.json").write_text(json.dumps(preparation, indent=2))
# 已核对全部 train/val/test 数量与图片引用；再对训练/验证图片做少量实际解码。
from task2_diet_record.data import DietDataset, read_annotations
for split in ("train", "val"):
    dataset = DietDataset(DATA_ROOT / "raw/annotations" / (split + ".jsonl"), DATA_ROOT, split)
    for index in range(min(8, len(dataset))):
        dataset[index]["image"].close()
    print(split, "前 8 张图片加载成功")
# test 在这里仅作静态完整性核对，标签不用于训练、选模、调参或提示词优化。
val_rows = read_annotations(DATA_ROOT / "raw/annotations/val.jsonl")[:2]
VAL_SMOKE = RUN_ROOT / "val_smoke.jsonl"
VAL_SMOKE.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in val_rows), encoding="utf-8")


## 核实并下载唯一指定模型
已查询官方仓库：必要文件约 516.6 MB，其中 safetensors 约 513.0 MB。先输出清单，再下载相同 revision 的模型与 Processor；复跑时缓存可复用。下载后核对大小和 LFS SHA256，不下载 Qwen/ONNX 等其他权重。


In [ ]:
run_module("task2_diet_record.prepare_model", "--revision", REVISION, "--destination", MODEL_DIR,
           "--manifest", RUN_ROOT / "model_manifest.json")
run_module("task2_diet_record.prepare_model", "--revision", REVISION, "--destination", MODEL_DIR,
           "--manifest", RUN_ROOT / "model_manifest.json", "--download")


## 未微调单图推理
使用验证集第一张真实图片，输入中没有 target。保存真实原文、解析/Schema 结果、时间和 GPU 显存峰值；JSON 不合法属于可记录的正常结果。256 token 上限可能导致截断，不能把该测试当作正式精度评价。


In [ ]:
run_module("task2_diet_record.integration", "--model-path", MODEL_DIR, "--data-root", DATA_ROOT,
           "--output", RUN_ROOT / "pretrained_inference.json", "--max-new-tokens", 256)
print((RUN_ROOT / "pretrained_inference.json").read_text())


## 真实 LoRA 更新、Checkpoint 和重载
复用 train.py：前 8 张 train 图片、batch 1、累积 1，先只更新 1 步。检查有限梯度、非零梯度、参数确实变化；保存断点后比较同一验证图片的确定性生成。验证图片和标签不会参与 backward。


In [ ]:
smoke_config = json.loads((REPO / "task2_diet_record/train_config.json").read_text())
smoke_config.update(epochs=1, batch_size=1, gradient_accumulation=1, max_new_tokens=256)
SMOKE_CONFIG = RUN_ROOT / "smoke_config.json"
SMOKE_CONFIG.write_text(json.dumps(smoke_config, indent=2))
COMMON = ["--config", SMOKE_CONFIG, "--model-path", MODEL_DIR, "--revision", REVISION,
          "--data-root", DATA_ROOT, "--device", "cuda", "--train-limit", 8, "--val-limit", 2,
          "--generate-val", 1, "--checkpoint-every", 1, "--verify-update", "--verify-reload"]
SMOKE_RUN = RUN_ROOT / "smoke_step1"
if not RESUME_CHECKPOINT:
    run_module("task2_diet_record.train", *COMMON, "--max-steps", 1, "--output", SMOKE_RUN)
    print((SMOKE_RUN / "steps.jsonl").read_text())
    print((SMOKE_RUN / "metrics.jsonl").read_text())
else:
    print("已指定旧断点，跳过新的第一步训练，直接恢复。")


## 实际验证恢复训练
恢复包括 LoRA、优化器、Python/NumPy/PyTorch/CUDA 随机状态、已完成步数和轮内样本游标。校验模型/配置/数据哈希；不完整断点会被拒绝。

默认接着刚保存的断点运行到**总计 2 步**。跨会话恢复时，可填 Drive 中的完整 checkpoint 路径，但必须使用相同 smoke_config、数据和模型；总步数上限必须大于断点步数。正式训练使用后面的独立配置和恢复开关。


In [ ]:
checkpoint = RESUME_CHECKPOINT or json.loads((SMOKE_RUN / "last_checkpoint.json").read_text())["path"]
from task2_diet_record.checkpoint import inspect_checkpoint
print("恢复游标:", inspect_checkpoint(checkpoint))
RESUME_RUN = RUN_ROOT / "smoke_resumed"
run_module("task2_diet_record.train", *COMMON, "--resume", checkpoint,
           "--max-steps", RESUME_TOTAL_STEPS, "--output", RESUME_RUN)
ADAPTER = Path(json.loads((RESUME_RUN / "last_checkpoint.json").read_text())["path"])
print((RESUME_RUN / "steps.jsonl").read_text())
print((RESUME_RUN / "metrics.jsonl").read_text())


## 验证集小样本生成评价
分别报告 JSON 解析率、Schema 合法率、有效预测 F1/MSE，以及约定失败补值方式的全样本指标。全零补值 MSE 不等于无条件的标准营养预测 MSE。匹配原料重量误差仅作诊断，不替代教师指标。没有合法输出时有效子集指标为 null。


In [ ]:
PREDICTIONS = RUN_ROOT / "val_predictions.jsonl"
run_module("task2_diet_record.predict", "--model-path", MODEL_DIR, "--adapter", ADAPTER,
           "--annotations", VAL_SMOKE, "--data-root", DATA_ROOT, "--device", "cuda",
           "--max-new-tokens", 256, "--output", PREDICTIONS)
run_module("evaluation.diet_metrics", "--annotations", VAL_SMOKE,
           "--predictions", PREDICTIONS, "--output", RUN_ROOT / "val_metrics.json")
print("请检查 pretrained_inference、smoke_step1、smoke_resumed 和 val_metrics 的真实报告。")


## 可选：正式训练（默认关闭）
确认上述 GPU 推理、更新、保存重载与恢复均成功后，再自行打开开关。默认正式参数仍为仓库配置的 3 epochs / batch 1 / 累积 8 / rank 8 / lr 1e-4，并非本阶段自动执行内容。T4 可先使用 float32 验证；正式耗时和显存必须依据实测。

每 100 步及每轮结束写入不可覆盖的完整断点，最新完整断点有 READY.json。保留全部断点，不自动清理。完整训练的 resume 必须使用相同配置，不能用 8 张样本的 smoke 断点恢复全量训练。


In [ ]:
RUN_FULL_TRAINING = False
FULL_RESUME_CHECKPOINT = ""
if RUN_FULL_TRAINING:
    full_args = ["--config", REPO / "task2_diet_record/train_config.json",
                 "--model-path", MODEL_DIR, "--revision", REVISION, "--data-root", DATA_ROOT,
                 "--device", "cuda", "--checkpoint-every", 100, "--output", RUN_ROOT / "full_train"]
    if FULL_RESUME_CHECKPOINT:
        full_args += ["--resume", FULL_RESUME_CHECKPOINT]
    run_module("task2_diet_record.train", *full_args)
else:
    print("未启动完整训练。下一步先检查两步集成测试的真实报告。")


正式模型选择完成后才运行完整 val 推理，固定方案后再使用 test 做最终测试。Notebook 不自动执行 test，不推送 Git，不输出未经运行的模型成绩。

参考：[Colab FAQ](https://research.google.com/colaboratory/faq.html)、[PEFT 保存格式](https://huggingface.co/docs/peft/developer_guides/checkpoint)、[官方模型](https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct)。
